# Review features

Turns the TMDB review text into numeric features.

Reads `data/tmdb_reviews.csv` and `review_lexicons.csv`.

One rule runs throughout: a listed word adds one, and a negator in the three tokens before
it flips that to minus one. Aspect scores come in pairs, how often the aspect is mentioned
and what the tone is when it is.

A score is left empty when the review gives nothing to score it from. That is different from
a score that came out neutral because words were present and cancelled.

## Load

In [ ]:
import re
import numpy as np
import pandas as pd

reviews = pd.read_csv("data/tmdb_reviews.csv")
lexicons = pd.read_csv("review_lexicons.csv")
by_dimension = lexicons.groupby("dimension")["token"].apply(set).to_dict()
reviews.shape

(27191, 7)

In [ ]:
TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)?")

NEGATORS = {
    "not", "no", "never", "none", "neither", "nor", "cannot", "without",
    "hardly", "barely", "scarcely", "don't", "doesn't", "didn't", "isn't",
    "aren't", "wasn't", "weren't", "won't", "wouldn't", "shouldn't",
    "couldn't", "hasn't", "haven't", "hadn't", "can't", "ain't",
}

token_lists = [TOKEN.findall(text.lower()) for text in reviews["content"].astype(str)]
pd.Series([len(t) for t in token_lists]).describe().round(1)

count    27191.0
mean       236.2
std        260.6
min          0.0
25%         80.0
50%        179.0
75%        309.0
max       4302.0
dtype: float64

## Scoring functions

`score_list` scores one list on its own, so positivity and negativity stay separate. Both can
go negative when a reviewer negates most of a list's words.

`score_tone` measures a window, so a positive word adds and a negative word subtracts.

In [ ]:
def score_list(tokens, positions):
    total = 0
    for i in positions:
        flipped = any(t in NEGATORS for t in tokens[max(0, i - 3):i])
        total += -1 if flipped else 1
    return total


def score_tone(tokens, positions):
    total = 0
    for i in positions:
        token = tokens[i]
        if token in by_dimension["positivity"]:
            total += -1 if any(t in NEGATORS for t in tokens[max(0, i - 3):i]) else 1
        elif token in by_dimension["negativity"]:
            total += 1 if any(t in NEGATORS for t in tokens[max(0, i - 3):i]) else -1
    return total

## Score every review

Each score is written only when there was something to score. A review with no positivity
word gets no positivity value at all, not a zero.

In [ ]:
ASPECTS = ["performance", "production", "effects", "humor", "aesthetics"]
SENTIMENT = by_dimension["positivity"] | by_dimension["negativity"]

rows = []

for tokens in token_lists:
    row = {"tokens": len(tokens)}

    positive = [i for i, t in enumerate(tokens) if t in by_dimension["positivity"]]
    negative = [i for i, t in enumerate(tokens) if t in by_dimension["negativity"]]
    row["positivity"] = score_list(tokens, positive) if positive else np.nan
    row["negativity"] = score_list(tokens, negative) if negative else np.nan

    for aspect in ASPECTS:
        positions = [i for i, t in enumerate(tokens) if t in by_dimension[aspect]]
        row[f"{aspect}_mentions"] = len(positions) if positions else np.nan

        window = set()
        for i in positions:
            window.update(range(max(0, i - 3), min(len(tokens), i + 4)))
        spoken = [i for i in window if tokens[i] in SENTIMENT]
        row[f"{aspect}_tone"] = score_tone(tokens, window) if spoken else np.nan

    rows.append(row)

features = pd.DataFrame(rows)
features.shape

(27191, 13)

## How much of each score is actually populated

In [ ]:
SCORES = ["positivity", "negativity"] + [f"{aspect}_{kind}" for aspect in ASPECTS
                                        for kind in ("mentions", "tone")]

pd.DataFrame({
    "populated": features[SCORES].notna().sum(),
    "empty": features[SCORES].isna().sum(),
    "populated_pct": (100 * features[SCORES].notna().mean()).round(1),
})

,populated,empty,populated_pct
positivity,24856,2335,91.4
negativity,15205,11986,55.9
performance_mentions,14367,12824,52.8
performance_tone,7248,19943,26.7
production_mentions,12375,14816,45.5
production_tone,4344,22847,16.0
effects_mentions,4915,22276,18.1
effects_tone,1662,25529,6.1
humor_mentions,6381,20810,23.5
humor_tone,3778,23413,13.9


## Adjust for length, then rescale to 0-1

Longer reviews collect more hits, so each score is multiplied by the ratio of the mean review
length to that review's own length. Empty scores stay empty.

Rescaling is not min-max. Most aspect scores are sparse, so min-max would bend the scale to
wherever the values happen to cluster. Instead each score is divided by its own 99th
percentile over the populated rows, and signed scores are centred on 0.5.

In [ ]:
SIGNED = ["positivity", "negativity"] + [f"{aspect}_tone" for aspect in ASPECTS]

length_ratio = features["tokens"].mean() / features["tokens"].clip(lower=1)

for column in SCORES:
    features[column] = features[column] * length_ratio
    ceiling = features[column].abs().quantile(0.99)
    ceiling = ceiling if ceiling and ceiling > 0 else 1.0
    clipped = features[column].clip(-ceiling, ceiling)
    if column in SIGNED:
        features[column] = 0.5 + 0.5 * clipped / ceiling
    else:
        features[column] = clipped.clip(lower=0) / ceiling

features[SCORES].describe().round(3)

,positivity,negativity,performance_mentions,performance_tone,production_mentions,production_tone,effects_mentions,effects_tone,humor_mentions,humor_tone,aesthetics_mentions,aesthetics_tone
count,24856.000,15205.000,14367.000,7248.000,12375.000,4344.000,4915.000,1662.000,6381.000,3778.000,6441.000,3965.000
mean,0.582,0.560,0.195,0.549,0.205,0.549,0.153,0.545,0.142,0.548,0.156,0.560
std,0.084,0.088,0.173,0.089,0.172,0.089,0.162,0.094,0.167,0.087,0.166,0.081
min,0.000,0.000,0.007,0.000,0.009,0.000,0.005,0.002,0.003,0.000,0.005,0.088
25%,0.533,0.519,0.081,0.515,0.093,0.516,0.061,0.512,0.044,0.512,0.062,0.521
50%,0.559,0.538,0.141,0.531,0.154,0.535,0.100,0.530,0.082,0.524,0.097,0.536
75%,0.601,0.575,0.244,0.565,0.253,0.566,0.180,0.562,0.167,0.553,0.182,0.569
max,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000,1.000


## Does the sentiment agree with the reviewer's own rating

In [ ]:
joined = reviews[["review_id", "tmdb_id", "title", "rating"]].join(features)
rated = joined.dropna(subset=["rating"])

pd.Series({
    "positivity": rated["positivity"].corr(rated["rating"]),
    "negativity": rated["negativity"].corr(rated["rating"]),
    "positivity_n": rated["positivity"].notna().sum(),
    "negativity_n": rated["negativity"].notna().sum(),
}).round(3)

positivity          0.257
negativity         -0.357
positivity_n    22950.000
negativity_n    14274.000
dtype: float64

## Save per review

In [ ]:
joined.to_csv("data/review_features.csv", index=False)
joined.shape

(27191, 17)

## Average each score by movie

One row per film. Each score is the mean over that film's reviews, skipping the reviews that
had nothing to score, so a film's average is computed only from reviews that actually carried
information for that score.

`reviews` is how many reviews the film has in total, and the count columns show how many of
them fed each score.

In [ ]:
SCORE_SET = SCORES

by_movie = joined.groupby(["tmdb_id", "title"]).agg(
    reviews=("review_id", "size"),
    mean_rating=("rating", "mean"),
    **{score: (score, "mean") for score in SCORE_SET},
    **{f"{score}_n": (score, "count") for score in SCORE_SET},
).reset_index()

by_movie.shape

(11509, 28)

In [ ]:
pd.DataFrame({
    "films_with_score": by_movie[SCORE_SET].notna().sum(),
    "films_total": len(by_movie),
    "coverage_pct": (100 * by_movie[SCORE_SET].notna().mean()).round(1),
})

,films_with_score,films_total,coverage_pct
positivity,10991,11509,95.5
negativity,8384,11509,72.8
performance_mentions,8082,11509,70.2
performance_tone,4918,11509,42.7
production_mentions,7448,11509,64.7
production_tone,3273,11509,28.4
effects_mentions,3109,11509,27.0
effects_tone,1266,11509,11.0
humor_mentions,4071,11509,35.4
humor_tone,2659,11509,23.1


In [ ]:
by_movie.to_csv("data/review_features_by_movie.csv", index=False)
by_movie.shape

(11509, 28)

In [ ]:
by_movie[["title", "reviews", "mean_rating", "positivity", "negativity",
          "performance_tone", "effects_tone"]].sort_values(
    "reviews", ascending=False).head(8)

,title,reviews,mean_rating,positivity,negativity,performance_tone,effects_tone
7208,Avengers: Endgame,60,8.583333,0.596181,0.596574,0.751342,0.630678
7393,Spider-Man: Into the Spider-Verse,53,6.750000,0.670172,0.549241,0.758523,0.705198
4250,The Avengers,41,8.062500,0.633197,0.549572,0.768981,0.524414
5972,Thor: The Dark World,40,7.000000,0.604828,0.679532,0.774068,0.550358
2209,Thor,40,7.571429,0.702988,0.523945,0.667113,0.679921
6174,Avengers: Age of Ultron,40,6.833333,0.625287,0.576722,0.582809,0.528138
7081,Thor: Ragnarok,38,7.714286,0.652618,0.673374,0.532092,NaN
317,Spider-Man,34,8.000000,0.648170,0.753623,0.985929,0.507629


## Top three films for each score

Ranked by the film average, highest first, counting only films with at least ten reviews so
that a single enthusiastic or hostile review cannot take a place. The number in brackets is
the review count behind the average.

In [63]:
qualified = by_movie[by_movie["reviews"] >= 10]
print(f"films with at least 10 reviews: {len(qualified)} of {len(by_movie)}")

board = {}

for score in SCORES:
    top = qualified.nlargest(3, score)
    values = [f"{title} {value:.2f} (n={int(n)})"
              for title, value, n in zip(top["title"], top[score], top["reviews"])]
    board[score] = values + [""] * (3 - len(values))

pd.DataFrame.from_dict(board, orient="index", columns=["1st", "2nd", "3rd"])

films with at least 10 reviews: 243 of 11509


,1st,2nd,3rd
positivity,Final Destination Bloodlines 0.76 (n=12),Assassin 33 A.D. 0.75 (n=19),Avengers: Infinity War 0.73 (n=33)
negativity,Killers of the Flower Moon 0.78 (n=12),Spider-Man 0.75 (n=34),War of the Worlds 0.68 (n=10)
performance_mentions,The Avengers 0.62 (n=41),Spider-Man 0.59 (n=34),Spider-Man 2 0.57 (n=32)
performance_tone,Lee Cronin's The Mummy 1.00 (n=11),Spider-Man 0.99 (n=34),Big Hero 6 0.85 (n=16)
production_mentions,Spider-Man: Into the Spider-Verse 0.69 (n=53),Thor 0.68 (n=40),Thor: Ragnarok 0.66 (n=38)
production_tone,Spider-Man: Far From Home 1.00 (n=16),The Beekeeper 1.00 (n=10),Final Destination Bloodlines 0.92 (n=12)
effects_mentions,Thor: Ragnarok 1.00 (n=38),Citizen Vigilante 1.00 (n=10),The Last Voyage of the Demeter 0.74 (n=10)
effects_tone,Spider-Man 2 1.00 (n=32),Dark Phoenix 1.00 (n=14),Guardians of the Galaxy 0.79 (n=15)
humor_mentions,Avengers: Age of Ultron 0.67 (n=40),Thor: Ragnarok 0.52 (n=38),Assassin 33 A.D. 0.39 (n=19)
humor_tone,Assassin 33 A.D. 0.69 (n=19),Guardians of the Galaxy 0.68 (n=15),Ant-Man 0.66 (n=12)


## Bottom three films for each score

Same ten-review floor, lowest first. For the signed scores a low value means the opposite
pole: least positive, most negative-toned, and so on.

In [ ]:
floor = {}

for score in SCORES:
    bottom = qualified.nsmallest(3, score)
    values = [f"{title} {value:.2f} (n={int(n)})"
              for title, value, n in zip(bottom["title"], bottom[score], bottom["reviews"])]
    floor[score] = values + [""] * (3 - len(values))

pd.DataFrame.from_dict(floor, orient="index", columns=["1st", "2nd", "3rd"])

,1st,2nd,3rd
positivity,The Mummy 0.53 (n=11),On Her Majesty's Secret Service 0.54 (n=10),F9 0.54 (n=11)
negativity,Inside Out 0.28 (n=11),Soul 0.44 (n=10),Jurassic World 0.48 (n=14)
performance_mentions,Mission: Impossible - The Final Reckoning 0.03...,Avatar: Fire and Ash 0.06 (n=14),Halloween 0.07 (n=10)
performance_tone,Spider-Man 3 0.37 (n=34),Prey 0.40 (n=13),Citizen Vigilante 0.45 (n=10)
production_mentions,Turning Red 0.06 (n=13),The Naked Gun 0.06 (n=11),The Flash 0.08 (n=11)
production_tone,Guardians of the Galaxy Vol. 2 0.24 (n=25),Citizen Vigilante 0.34 (n=10),Thor: Ragnarok 0.41 (n=38)
effects_mentions,Jojo Rabbit 0.01 (n=13),Once Upon a Time... in Hollywood 0.01 (n=15),Joker 0.02 (n=19)
effects_tone,Disclosure Day 0.29 (n=16),Wonder Woman 1984 0.34 (n=17),Gladiator II 0.38 (n=15)
humor_mentions,Tenet 0.01 (n=18),Godzilla: King of the Monsters 0.01 (n=10),Terminator: Dark Fate 0.01 (n=12)
humor_tone,Captain America: Civil War 0.37 (n=25),Disclosure Day 0.42 (n=16),Birds of Prey (and the Fantabulous Emancipatio...
